In [6]:
import pandas as pd
import numpy as np
import ict_feature_functions as ict_fct
from IPython.display import display
from pathlib import Path


In [7]:
file_path="c:\\Users\\bessa\\Documents\\forexdata\\FX_Dukascopy_5y\\FX_ICT_Enriched\\EURUSD.csv"
df=pd.read_csv(file_path)

In [8]:
#first table
# hours and min
df=ict_fct.compute_utc_time_features(df, "timestamp")
# day of the week utc
df=ict_fct.compute_day_of_week(df, "timestamp")
#session 
df = ict_fct.compute_session(df)  
#killzone 
df = ict_fct.compute_killzone(df)

cols=[
    "timestamp",
    "hour",
    "minute",
    "day_of_week",
    "session",
    "killzone",]
display(df[cols].head(5))
display(df[cols].iloc[1920:1926])  

,timestamp,hour,minute,day_of_week,session,killzone
0,2020-11-23T00:00:00Z,0,0,0,Asia,None
1,2020-11-23T00:01:00Z,0,1,0,Asia,None
2,2020-11-23T00:02:00Z,0,2,0,Asia,None
3,2020-11-23T00:03:00Z,0,3,0,Asia,None
4,2020-11-23T00:04:00Z,0,4,0,Asia,None


,timestamp,hour,minute,day_of_week,session,killzone
1920,2020-11-24T08:00:00Z,8,0,1,London,London_KZ
1921,2020-11-24T08:01:00Z,8,1,1,London,London_KZ
1922,2020-11-24T08:02:00Z,8,2,1,London,London_KZ
1923,2020-11-24T08:03:00Z,8,3,1,London,London_KZ
1924,2020-11-24T08:04:00Z,8,4,1,London,London_KZ
1925,2020-11-24T08:05:00Z,8,5,1,London,London_KZ


In [9]:
display(df["session"].value_counts(),
df["killzone"].value_counts())


session
Asia          626400
Afterhours    547980
London        391500
NewYork       313200
Name: count, dtype: int64

killzone
London_KZ     236205
NewYork_KZ    157905
Name: count, dtype: int64

In [13]:
# convert utc to timestamp_ny, hour_ny, minute_ny
df = ict_fct.add_timestamp_ny(df) 
# session_ny (Tokyo/London/NY by NY time)
df = ict_fct.compute_session_ny(df)     
# killzone new york       
df = ict_fct.compute_killzone_ny(df) 
# is_ny_day_open, is_sunday_open_ny, is_ny_midnight          
df = ict_fct.add_ny_day_week_flags(df) 
#day of week new york        
df=ict_fct.compute_day_of_week_ny(df)
# example 

cols = [
    "timestamp",  
    "timestamp_ny",
    "hour_ny",
    "minute_ny",
    "session_ny",
    "killzone_ny",
    "is_ny_day_open",
    "is_sunday_open_ny",
    "is_ny_midnight",
    "day_of_week_ny",
]

display(df[cols].head(5),df[cols].iloc[1920:1926])


,timestamp,timestamp_ny,hour_ny,minute_ny,session_ny,killzone_ny,is_ny_day_open,is_sunday_open_ny,is_ny_midnight,day_of_week_ny
0,2020-11-23T00:00:00Z,2020-11-22 19:00:00-05:00,19,0,Tokyo,None,False,False,False,6
1,2020-11-23T00:01:00Z,2020-11-22 19:01:00-05:00,19,1,Tokyo,None,False,False,False,6
2,2020-11-23T00:02:00Z,2020-11-22 19:02:00-05:00,19,2,Tokyo,None,False,False,False,6
3,2020-11-23T00:03:00Z,2020-11-22 19:03:00-05:00,19,3,Tokyo,None,False,False,False,6
4,2020-11-23T00:04:00Z,2020-11-22 19:04:00-05:00,19,4,Tokyo,None,False,False,False,6


,timestamp,timestamp_ny,hour_ny,minute_ny,session_ny,killzone_ny,is_ny_day_open,is_sunday_open_ny,is_ny_midnight,day_of_week_ny
1920,2020-11-24T08:00:00Z,2020-11-24 03:00:00-05:00,3,0,London,London_KZ,False,False,False,1
1921,2020-11-24T08:01:00Z,2020-11-24 03:01:00-05:00,3,1,London,London_KZ,False,False,False,1
1922,2020-11-24T08:02:00Z,2020-11-24 03:02:00-05:00,3,2,London,London_KZ,False,False,False,1
1923,2020-11-24T08:03:00Z,2020-11-24 03:03:00-05:00,3,3,London,London_KZ,False,False,False,1
1924,2020-11-24T08:04:00Z,2020-11-24 03:04:00-05:00,3,4,London,London_KZ,False,False,False,1
1925,2020-11-24T08:05:00Z,2020-11-24 03:05:00-05:00,3,5,London,London_KZ,False,False,False,1


In [16]:
print(df["is_sunday_open_ny"].value_counts(),
df["is_ny_day_open"].value_counts(),df["is_ny_midnight"].value_counts())

is_sunday_open_ny
False    1879080
Name: count, dtype: int64 is_ny_day_open
False    1877776
True        1304
Name: count, dtype: int64 is_ny_midnight
False    1877775
True        1305
Name: count, dtype: int64


is_sunday_open_ny is always False in this dataset because project 1 removed all weekend bars (Saturday and Sunday).
As a result there is no bar at the New York Sunday open (Sunday 17:00 NY), so this flag never occurs.

In [17]:
#second table 
df = ict_fct.compute_high_lookback(df)
df = ict_fct.compute_low_lookback(df)
df = ict_fct.compute_price_above_HTF_mid(df)
df=  ict_fct.compute_sweep_HTF_high(df)
df = ict_fct.compute_sweep_HTF_low(df)
cols=[
    "timestamp",
    "HTF_high_lookback",
    "HTF_low_lookback",
    "price_above_HTF_mid",
    "sweep_HTF_high",
    "sweep_HTF_low",    
]
display(df[cols].head(5),df[cols].iloc[61:66])

,timestamp,HTF_high_lookback,HTF_low_lookback,price_above_HTF_mid,sweep_HTF_high,sweep_HTF_low
0,2020-11-23T00:00:00Z,NaN,NaN,False,False,False
1,2020-11-23T00:01:00Z,NaN,NaN,False,False,False
2,2020-11-23T00:02:00Z,NaN,NaN,False,False,False
3,2020-11-23T00:03:00Z,NaN,NaN,False,False,False
4,2020-11-23T00:04:00Z,NaN,NaN,False,False,False


,timestamp,HTF_high_lookback,HTF_low_lookback,price_above_HTF_mid,sweep_HTF_high,sweep_HTF_low
61,2020-11-23T01:01:00Z,1.18734,1.18636,True,False,False
62,2020-11-23T01:02:00Z,1.18734,1.18636,True,False,False
63,2020-11-23T01:03:00Z,1.18734,1.18636,True,False,False
64,2020-11-23T01:04:00Z,1.18734,1.18638,True,False,False
65,2020-11-23T01:05:00Z,1.18734,1.18638,True,False,False


Table 2 – HTF range and sweeps (uses past 60 minutes, first 60 rows will be NaN)
and starting from row 66 we can see values 

In [19]:
# 1) Compute swings first
df = ict_fct.compute_swing_high(df)
df = ict_fct.compute_swing_low(df)

# 2) Map each pair to its pip size
PIP_SIZE_BY_PAIR = {
    "EURUSD": 0.0001,
    "GBPUSD": 0.0001,
    "AUDUSD": 0.0001,
    "USDCHF": 0.0001,
    "USDCAD": 0.0001,
    "USDJPY": 0.01,
}

# 3) Infer the pair name from the filename
pair = Path(file_path).stem   # "EURUSD", "USDJPY", ...
pip_size = PIP_SIZE_BY_PAIR[pair]

# 4) Compute equal highs/lows with the right pip size
df = ict_fct.compute_equal_high_low(df, pip_size=pip_size)

# 5) Inspect a few rows
cols = [
    "timestamp",
    "swing_high",
    "swing_low",
    "equal_high",
    "equal_low",
]
display(df[cols].head(5))


,timestamp,swing_high,swing_low,equal_high,equal_low
0,2020-11-23T00:00:00Z,False,False,False,False
1,2020-11-23T00:01:00Z,False,False,False,False
2,2020-11-23T00:02:00Z,False,False,False,False
3,2020-11-23T00:03:00Z,False,True,False,False
4,2020-11-23T00:04:00Z,False,False,False,False


In [20]:
display(df["swing_high"].value_counts(),
df["swing_low"].value_counts(),
df["equal_high"].value_counts(),
df["equal_low"].value_counts())

swing_high
False    1737090
True      141990
Name: count, dtype: int64

swing_low
False    1739967
True      139113
Name: count, dtype: int64

equal_high
False    1799974
True       79106
Name: count, dtype: int64

equal_low
False    1803150
True       75930
Name: count, dtype: int64

In [21]:
df = ict_fct.compute_displacement_up(df)
df = ict_fct.compute_displacement_down(df)
df = ict_fct.compute_range_size(df)
df = ict_fct.compute_avg_range_20(df)
df = ict_fct.compute_range_expansion(df)
cols=[
    "timestamp",
    "displacement_up",
    "displacement_down",
    "range_size",
    "avg_range_20",  
    "range_expansion"
]
display(df[cols].head(5))
display(df[cols].iloc[50:56])

,timestamp,displacement_up,displacement_down,range_size,avg_range_20,range_expansion
0,2020-11-23T00:00:00Z,False,False,0.00013,NaN,False
1,2020-11-23T00:01:00Z,False,False,0.00007,NaN,False
2,2020-11-23T00:02:00Z,False,False,0.00014,NaN,False
3,2020-11-23T00:03:00Z,False,False,0.00007,NaN,False
4,2020-11-23T00:04:00Z,False,False,0.00008,NaN,False


,timestamp,displacement_up,displacement_down,range_size,avg_range_20,range_expansion
50,2020-11-23T00:50:00Z,False,False,0.00002,0.000067,False
51,2020-11-23T00:51:00Z,False,False,0.00002,0.000065,False
52,2020-11-23T00:52:00Z,False,False,0.00000,0.000064,False
53,2020-11-23T00:53:00Z,True,False,0.00010,0.000060,True
54,2020-11-23T00:54:00Z,False,False,0.00007,0.000062,False
55,2020-11-23T00:55:00Z,False,True,0.00010,0.000063,True


In [23]:
#fifth table
df = ict_fct.compute_prev_day_levels(df)#high and low
df =  ict_fct.compute_prev_week_levels(df)#high and low

cols=[
    "timestamp",
    "prev_day_high",
    "prev_day_low",
    "prev_week_high",
    "prev_week_low",
]
display(df[cols].head(5))
display(df[cols].iloc[200:206])


,timestamp,prev_day_high,prev_day_low,prev_week_high,prev_week_low
0,2020-11-23T00:00:00Z,NaN,NaN,NaN,NaN
1,2020-11-23T00:01:00Z,NaN,NaN,NaN,NaN
2,2020-11-23T00:02:00Z,NaN,NaN,NaN,NaN
3,2020-11-23T00:03:00Z,NaN,NaN,NaN,NaN
4,2020-11-23T00:04:00Z,NaN,NaN,NaN,NaN


,timestamp,prev_day_high,prev_day_low,prev_week_high,prev_week_low
200,2020-11-23T03:20:00Z,NaN,NaN,NaN,NaN
201,2020-11-23T03:21:00Z,NaN,NaN,NaN,NaN
202,2020-11-23T03:22:00Z,NaN,NaN,NaN,NaN
203,2020-11-23T03:23:00Z,NaN,NaN,NaN,NaN
204,2020-11-23T03:24:00Z,NaN,NaN,NaN,NaN
205,2020-11-23T03:25:00Z,NaN,NaN,NaN,NaN


In [24]:
display(df["prev_day_high"].value_counts(),
df["prev_day_low"].value_counts(),
df["prev_week_high"].value_counts(),   
df["prev_week_low"].value_counts())

prev_day_high
1.08713    4320
1.07987    2880
1.17427    2880
1.07344    2880
1.09171    2880
           ... 
1.16224    1440
1.16213    1440
1.16490    1440
1.16547    1440
1.15499    1320
Name: count, Length: 1261, dtype: int64

prev_day_low
1.09000    4320
1.13031    4320
1.05328    4320
1.04826    2880
1.20560    2880
           ... 
1.15853    1440
1.16008    1440
1.16176    1440
1.16255    1440
1.15013    1320
Name: count, Length: 1261, dtype: int64

prev_week_high
1.14948    14400
1.19641     7200
1.21773     7200
1.21662     7200
1.11462     7200
           ...  
1.21132     7200
1.19899     7200
1.19888     7200
1.15910     7200
1.16559     7080
Name: count, Length: 259, dtype: int64

prev_week_low
1.05057    14400
1.03496    14400
1.07658    14400
1.15566     7200
1.14461     7200
           ...  
1.21155     7200
1.20587     7200
1.19236     7200
1.17997     7200
1.15409     7080
Name: count, Length: 257, dtype: int64